In [1]:
!ls

__notebook__.ipynb


In [2]:
!pip install ../input/mtcnn-package/mtcnn-0.1.0-py3-none-any.whl

Processing /kaggle/input/mtcnn-package/mtcnn-0.1.0-py3-none-any.whl


In [3]:
import sys
import numpy as np
import pandas as pd
import cv2
from keras.optimizers import Adam, SGD
from tqdm.notebook import tqdm
import glob
from mtcnn import MTCNN

# use shutil to copy file from input folder to working folder
#from shutil import copyfile

Using TensorFlow backend.


In [4]:
# -*- coding:utf-8 -*-

from keras.models import Model as KerasModel
from keras.layers import Input, Dense, Flatten, Conv2D, MaxPooling2D, BatchNormalization, Dropout, Reshape, Concatenate, LeakyReLU
from keras.optimizers import Adam

IMGWIDTH = 256

class Classifier:
    def __init__():
        self.model = 0
    
    def predict(self, x):
        return self.model.predict(x)
    
    def fit(self, x, y):
        return self.model.train_on_batch(x, y)
    
    def get_accuracy(self, x, y):
        return self.model.test_on_batch(x, y)
    
    def load(self, path):
        self.model.load_weights(path)


class Meso1(Classifier):
    """
    Feature extraction + Classification
    """
    def __init__(self, learning_rate = 0.001, dl_rate = 1):
        self.model = self.init_model(dl_rate)
        optimizer = Adam(lr = learning_rate)
        self.model.compile(optimizer = optimizer, loss = 'mean_squared_error', metrics = ['accuracy'])
    
    def init_model(self, dl_rate):
        x = Input(shape = (IMGWIDTH, IMGWIDTH, 3))
        
        x1 = Conv2D(16, (3, 3), dilation_rate = dl_rate, strides = 1, padding='same', activation = 'relu')(x)
        x1 = Conv2D(4, (1, 1), padding='same', activation = 'relu')(x1)
        x1 = BatchNormalization()(x1)
        x1 = MaxPooling2D(pool_size=(8, 8), padding='same')(x1)

        y = Flatten()(x1)
        y = Dropout(0.5)(y)
        y = Dense(1, activation = 'sigmoid')(y)
        return KerasModel(inputs = x, outputs = y)


class Meso4(Classifier):
    def __init__(self, learning_rate = 0.001):
        self.model = self.init_model()
        optimizer = Adam(lr = learning_rate)
        self.model.compile(optimizer = optimizer, loss = 'mean_squared_error', metrics = ['accuracy'])
    
    def init_model(self): 
        x = Input(shape = (IMGWIDTH, IMGWIDTH, 3))
        
        x1 = Conv2D(8, (3, 3), padding='same', activation = 'relu')(x)
        x1 = BatchNormalization()(x1)
        x1 = MaxPooling2D(pool_size=(2, 2), padding='same')(x1)
        
        x2 = Conv2D(8, (5, 5), padding='same', activation = 'relu')(x1)
        x2 = BatchNormalization()(x2)
        x2 = MaxPooling2D(pool_size=(2, 2), padding='same')(x2)
        
        x3 = Conv2D(16, (5, 5), padding='same', activation = 'relu')(x2)
        x3 = BatchNormalization()(x3)
        x3 = MaxPooling2D(pool_size=(2, 2), padding='same')(x3)
        
        x4 = Conv2D(16, (5, 5), padding='same', activation = 'relu')(x3)
        x4 = BatchNormalization()(x4)
        x4 = MaxPooling2D(pool_size=(4, 4), padding='same')(x4)
        
        y = Flatten()(x4)
        y = Dropout(0.5)(y)
        y = Dense(16)(y)
        y = LeakyReLU(alpha=0.1)(y)
        y = Dropout(0.5)(y)
        y = Dense(1, activation = 'sigmoid')(y)

        return KerasModel(inputs = x, outputs = y)


class MesoInception4(Classifier):
    def __init__(self, learning_rate = 0.001):
        self.model = self.init_model()
        optimizer = Adam(lr = learning_rate)
        self.model.compile(optimizer = optimizer, loss = 'mean_squared_error', metrics = ['accuracy'])
    
    def InceptionLayer(self, a, b, c, d):
        def func(x):
            x1 = Conv2D(a, (1, 1), padding='same', activation='relu')(x)
            
            x2 = Conv2D(b, (1, 1), padding='same', activation='relu')(x)
            x2 = Conv2D(b, (3, 3), padding='same', activation='relu')(x2)
            
            x3 = Conv2D(c, (1, 1), padding='same', activation='relu')(x)
            x3 = Conv2D(c, (3, 3), dilation_rate = 2, strides = 1, padding='same', activation='relu')(x3)
            
            x4 = Conv2D(d, (1, 1), padding='same', activation='relu')(x)
            x4 = Conv2D(d, (3, 3), dilation_rate = 3, strides = 1, padding='same', activation='relu')(x4)

            y = Concatenate(axis = -1)([x1, x2, x3, x4])
            
            return y
        return func
    
    def init_model(self):
        x = Input(shape = (IMGWIDTH, IMGWIDTH, 3))
        
        x1 = self.InceptionLayer(1, 4, 4, 2)(x)
        x1 = BatchNormalization()(x1)
        x1 = MaxPooling2D(pool_size=(2, 2), padding='same')(x1)
        
        x2 = self.InceptionLayer(2, 4, 4, 2)(x1)
        x2 = BatchNormalization()(x2)
        x2 = MaxPooling2D(pool_size=(2, 2), padding='same')(x2)        
        
        x3 = Conv2D(16, (5, 5), padding='same', activation = 'relu')(x2)
        x3 = BatchNormalization()(x3)
        x3 = MaxPooling2D(pool_size=(2, 2), padding='same')(x3)
        
        x4 = Conv2D(16, (5, 5), padding='same', activation = 'relu')(x3)
        x4 = BatchNormalization()(x4)
        x4 = MaxPooling2D(pool_size=(4, 4), padding='same')(x4)
        
        y = Flatten()(x4)
        y = Dropout(0.5)(y)
        y = Dense(16)(y)
        y = LeakyReLU(alpha=0.1)(y)
        y = Dropout(0.5)(y)
        y = Dense(1, activation = 'sigmoid')(y)

        return KerasModel(inputs = x, outputs = y)

In [5]:
"""""# -*- coding:utf-8 -*-
# Darius' approach -This generates a batch of nearly 50 frames per scene. And makes predictions on these frames.But the kernel goes out after using it. 

import random
from os import listdir
from os.path import isfile, join

import numpy as np
from math import floor
from scipy.ndimage.interpolation import zoom, rotate

import imageio
import face_recognition


## Face extraction

class Video:
    def __init__(self, path):
        self.path = path
        self.container = imageio.get_reader(path, 'ffmpeg')
        self.length = self.container.count_frames()
        self.fps = self.container.get_meta_data()['fps']
    
    def init_head(self):
        self.container.set_image_index(0)
    
    def next_frame(self):
        self.container.get_next_data()
    
    def get(self, key):
        return self.container.get_data(key)
    
    def __call__(self, key):
        return self.get(key)
    
    def __len__(self):
        return self.length


class FaceFinder(Video):
    def __init__(self, path, load_first_face = True):
        super().__init__(path)
        self.faces = {}
        self.coordinates = {}  # stores the face (locations center, rotation, length)
        self.last_frame = self.get(0)
        self.frame_shape = self.last_frame.shape[:2]
        self.last_location = (0, 200, 200, 0)
        if (load_first_face):
            face_positions = face_recognition.face_locations(self.last_frame, number_of_times_to_upsample=2)
            if len(face_positions) > 0:
                self.last_location = face_positions[0]
    
    def load_coordinates(self, filename):
        np_coords = np.load(filename)
        self.coordinates = np_coords.item()
    
    def expand_location_zone(self, loc, margin = 0.2):
        ''' Adds a margin around a frame slice '''
        offset = round(margin * (loc[2] - loc[0]))
        y0 = max(loc[0] - offset, 0)
        x1 = min(loc[1] + offset, self.frame_shape[1])
        y1 = min(loc[2] + offset, self.frame_shape[0])
        x0 = max(loc[3] - offset, 0)
        return (y0, x1, y1, x0)
    
    @staticmethod
    def upsample_location(reduced_location, upsampled_origin, factor):
        ''' Adapt a location to an upsampled image slice '''
        y0, x1, y1, x0 = reduced_location
        Y0 = round(upsampled_origin[0] + y0 * factor)
        X1 = round(upsampled_origin[1] + x1 * factor)
        Y1 = round(upsampled_origin[0] + y1 * factor)
        X0 = round(upsampled_origin[1] + x0 * factor)
        return (Y0, X1, Y1, X0)

    @staticmethod
    def pop_largest_location(location_list):
        max_location = location_list[0]
        max_size = 0
        if len(location_list) > 1:
            for location in location_list:
                size = location[2] - location[0]
                if size > max_size:
                    max_size = size
                    max_location = location
        return max_location
    
    @staticmethod
    def L2(A, B):
        return np.sqrt(np.sum(np.square(A - B)))
    
    def find_coordinates(self, landmark, K = 2.2):
        '''
        We either choose K * distance(eyes, mouth),
        or, if the head is tilted, K * distance(eye 1, eye 2)
        /!\ landmarks coordinates are in (x,y) not (y,x)
        '''
        E1 = np.mean(landmark['left_eye'], axis=0)
        E2 = np.mean(landmark['right_eye'], axis=0)
        E = (E1 + E2) / 2
        N = np.mean(landmark['nose_tip'], axis=0) / 2 + np.mean(landmark['nose_bridge'], axis=0) / 2
        B1 = np.mean(landmark['top_lip'], axis=0)
        B2 = np.mean(landmark['bottom_lip'], axis=0)
        B = (B1 + B2) / 2

        C = N
        l1 = self.L2(E1, E2)
        l2 = self.L2(B, E)
        l = max(l1, l2) * K
        if (B[1] == E[1]):
            if (B[0] > E[0]):
                rot = 90
            else:
                rot = -90
        else:
            rot = np.arctan((B[0] - E[0]) / (B[1] - E[1])) / np.pi * 180
        
        return ((floor(C[1]), floor(C[0])), floor(l), rot)
    
    
    def find_faces(self, resize = 0.5, stop = 0, skipstep = 0, no_face_acceleration_threshold = 3, cut_left = 0, cut_right = -1, use_frameset = False, frameset = []):
        '''
        The core function to extract faces from frames
        using previous frame location and downsampling to accelerate the loop.
        '''
        not_found = 0
        no_face = 0
        no_face_acc = 0
        
        # to only deal with a subset of a video, for instance I-frames only
        if (use_frameset):
            finder_frameset = frameset
        else:
            if (stop != 0):
                finder_frameset = range(0, min(self.length, stop), skipstep + 1)
            else:
                finder_frameset = range(0, self.length, skipstep + 1)
        
        # Quick face finder loop
        for i in finder_frameset:
            # Get frame
            frame = self.get(i)
            if (cut_left != 0 or cut_right != -1):
                frame[:, :cut_left] = 0
                frame[:, cut_right:] = 0            
            
            # Find face in the previously found zone
            potential_location = self.expand_location_zone(self.last_location)
            potential_face_patch = frame[potential_location[0]:potential_location[2], potential_location[3]:potential_location[1]]
            potential_face_patch_origin = (potential_location[0], potential_location[3])
    
            reduced_potential_face_patch = zoom(potential_face_patch, (resize, resize, 1))
            reduced_face_locations = face_recognition.face_locations(reduced_potential_face_patch, model = 'cnn')
            
            if len(reduced_face_locations) > 0:
                no_face_acc = 0  # reset the no_face_acceleration mode accumulator

                reduced_face_location = self.pop_largest_location(reduced_face_locations)
                face_location = self.upsample_location(reduced_face_location,
                                                    potential_face_patch_origin,
                                                    1 / resize)
                self.faces[i] = face_location
                self.last_location = face_location
                
                # extract face rotation, length and center from landmarks
                landmarks = face_recognition.face_landmarks(frame, [face_location])
                if len(landmarks) > 0:
                    # we assume that there is one and only one landmark group
                    self.coordinates[i] = self.find_coordinates(landmarks[0])
            else:
                not_found += 1

                if no_face_acc < no_face_acceleration_threshold:
                    # Look for face in full frame
                    face_locations = face_recognition.face_locations(frame, number_of_times_to_upsample = 2)
                else:
                    # Avoid spending to much time on a long scene without faces
                    reduced_frame = zoom(frame, (resize, resize, 1))
                    face_locations = face_recognition.face_locations(reduced_frame)
                    
                if len(face_locations) > 0:
                    print('Face extraction warning : ', i, '- found face in full frame', face_locations)
                    no_face_acc = 0  # reset the no_face_acceleration mode accumulator
                    
                    face_location = self.pop_largest_location(face_locations)
                    
                    # if was found on a reduced frame, upsample location
                    if no_face_acc > no_face_acceleration_threshold:
                        face_location = self.upsample_location(face_location, (0, 0), 1 / resize)
                    
                    self.faces[i] = face_location
                    self.last_location = face_location
                    
                    # extract face rotation, length and center from landmarks
                    landmarks = face_recognition.face_landmarks(frame, [face_location])
                    if len(landmarks) > 0:
                        self.coordinates[i] = self.find_coordinates(landmarks[0])
                else:
                    print('Face extraction warning : ',i, '- no face')
                    no_face_acc += 1
                    no_face += 1

        print('Face extraction report of', 'not_found :', not_found)
        print('Face extraction report of', 'no_face :', no_face)
        return 0
    
    def get_face(self, i):
        ''' Basic unused face extraction without alignment '''
        frame = self.get(i)
        if i in self.faces:
            loc = self.faces[i]
            patch = frame[loc[0]:loc[2], loc[3]:loc[1]]
            return patch
        return frame
    
    @staticmethod
    def get_image_slice(img, y0, y1, x0, x1):
        '''Get values outside the domain of an image'''
        m, n = img.shape[:2]
        padding = max(-y0, y1-m, -x0, x1-n, 0)
        padded_img = np.pad(img, ((padding, padding), (padding, padding), (0, 0)), 'reflect')
        return padded_img[(padding + y0):(padding + y1),
                        (padding + x0):(padding + x1)]
    
    def get_aligned_face(self, i, l_factor = 1.3):
        '''
        The second core function that converts the data from self.coordinates into an face image.
        '''
        frame = self.get(i)
        if i in self.coordinates:
            c, l, r = self.coordinates[i]
            l = int(l) * l_factor # fine-tuning the face zoom we really want
            dl_ = floor(np.sqrt(2) * l / 2) # largest zone even when rotated
            patch = self.get_image_slice(frame,
                                    floor(c[0] - dl_),
                                    floor(c[0] + dl_),
                                    floor(c[1] - dl_),
                                    floor(c[1] + dl_))
            rotated_patch = rotate(patch, -r, reshape=False)
            # note : dl_ is the center of the patch of length 2dl_
            return self.get_image_slice(rotated_patch,
                                    floor(dl_-l//2),
                                    floor(dl_+l//2),
                                    floor(dl_-l//2),
                                    floor(dl_+l//2))
        return frame


## Face prediction

class FaceBatchGenerator:
    '''
    Made to deal with framesubsets of video.
    '''
    def __init__(self, face_finder, target_size = 256):
        self.finder = face_finder
        self.target_size = target_size
        self.head = 0
        self.length = int(face_finder.length)

    def resize_patch(self, patch):
        m, n = patch.shape[:2]
        return zoom(patch, (self.target_size / m, self.target_size / n, 1))
    
    def next_batch(self, batch_size = 50):
        batch = np.zeros((1, self.target_size, self.target_size, 3))
        stop = min(self.head + batch_size, self.length)
        i = 0
        while (i < batch_size) and (self.head < self.length):
            if self.head in self.finder.coordinates:
                patch = self.finder.get_aligned_face(self.head)
                batch = np.concatenate((batch, np.expand_dims(self.resize_patch(patch), axis = 0)),
                                        axis = 0)
                i += 1
            self.head += 1
        return batch[1:]


def predict_faces(generator, classifier, batch_size = 50, output_size = 1):
    '''
    Compute predictions for a face batch generator
    '''
    n = len(generator.finder.coordinates.items())
    profile = np.zeros((1, output_size))
    for epoch in range(n // batch_size + 1):
        face_batch = generator.next_batch(batch_size = batch_size)
        prediction = classifier.predict(face_batch)
        if (len(prediction) > 0):
            profile = np.concatenate((profile, prediction))
    return profile[1:]


def compute_accuracy(classifier, dirname, frame_subsample_count = 30):
    '''
    Extraction + Prediction over a video
    '''
    filenames = [f for f in listdir(dirname) if isfile(join(dirname, f)) and ((f[-4:] == '.mp4') or (f[-4:] == '.avi') or (f[-4:] == '.mov'))]
    predictions = {}
    
    for vid in filenames:
        print('Dealing with video ', vid)
        
        # Compute face locations and store them in the face finder
        face_finder = FaceFinder(join(dirname, vid), load_first_face = False)
        skipstep = max(floor(face_finder.length / frame_subsample_count), 0)
        face_finder.find_faces(resize=0.5, skipstep = skipstep)
        
        print('Predicting ', vid)
        gen = FaceBatchGenerator(face_finder)
        p = predict_faces(gen, classifier)
        
        predictions[vid[:-4]] = (np.mean(p > 0.5), p)
    return predictions"""""

'""# -*- coding:utf-8 -*-\n# Darius\' approach -This generates a batch of nearly 50 frames per scene. And makes predictions on these frames.But the kernel goes out after using it. \n\nimport random\nfrom os import listdir\nfrom os.path import isfile, join\n\nimport numpy as np\nfrom math import floor\nfrom scipy.ndimage.interpolation import zoom, rotate\n\nimport imageio\nimport face_recognition\n\n\n## Face extraction\n\nclass Video:\n    def __init__(self, path):\n        self.path = path\n        self.container = imageio.get_reader(path, \'ffmpeg\')\n        self.length = self.container.count_frames()\n        self.fps = self.container.get_meta_data()[\'fps\']\n    \n    def init_head(self):\n        self.container.set_image_index(0)\n    \n    def next_frame(self):\n        self.container.get_next_data()\n    \n    def get(self, key):\n        return self.container.get_data(key)\n    \n    def __call__(self, key):\n        return self.get(key)\n    \n    def __len__(self):\n       

In [6]:
INPUT = "/kaggle/input/deepfake-detection-challenge"
MAX_SKIP = 10
NUM_FRAME= 150
LR_ALPHA = 1e-3

In [7]:
import os
test_dir = os.path.join(INPUT, "test_videos/")
filenames = os.listdir(test_dir)
prediction_filenames = filenames
test_video_files = [os.path.join(test_dir, fn) for fn in filenames]

print(test_video_files[:2])

['/kaggle/input/deepfake-detection-challenge/test_videos/nymodlmxni.mp4', '/kaggle/input/deepfake-detection-challenge/test_videos/oaguiggjyv.mp4']


In [8]:
detector = MTCNN()

def detect_face(image):
    # convert to RGB for MTCNN use
    img = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    final = []
    detected_faces_raw = detector.detect_faces(img)

    # check if any faces detected
    if not detected_faces_raw:
        #print("no faces")
        return []
    
    # store confidences & bounding box
    confidences = []
    for n in detected_faces_raw:
        x, y, w, h = n["box"]
        final.append([x, y, w, h])
        confidences.append(n["confidence"])
        pass
    # check if detector is confident that this is a face
    if max(confidences) < 0.7:
        return []
    # get the most confident detected face bounding box
    max_conf_coord = final[confidences.index(max(confidences))]
    
    return max_conf_coord


def crop(image, x, y, w, h):
    # enlarge bounding box to ensure containing the fucking face
    x -= 40
    y -= 40
    w += 80
    h += 80
    if x < 0: x = 0
    if y <= 0: y = 0
    
    crop = image[y : y + h, x : x + w, :]
    crop = cv2.resize(crop, (256, 256))
    return cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

def detect_video(video):
    # extract the middle frame of a video
    camera = cv2.VideoCapture(video)
    camera.set(1, NUM_FRAME)
    success, vframe = camera.read()
    
    # convert color to RGB & detect face
    vframe = cv2.cvtColor(vframe, cv2.COLOR_BGR2RGB)
    bounding_box = detect_face(vframe)
    
    # deal with no faces, try the following 10 frames to detect bounding box
    if not bounding_box:
        count = 0
        current = NUM_FRAME
        
        for i in range(NUM_FRAME + 1, NUM_FRAME + 11):
            success, vframe = camera.read()
            vframe = cv2.cvtColor(vframe, cv2.COLOR_BGR2RGB)
            bounding_box = detect_face(vframe)
            # check if getting a face
            if len(bounding_box) != 0:
                break
        # check if still no face detected
        if not bounding_box:
            print("no face found")
            prediction_filenames.remove(video.replace(test_dir,''))
            return None
    
    # got a face
    x, y, w, h = bounding_box
    camera.release()
    return crop(vframe, x, y, w, h)

In [9]:
!ls

__notebook__.ipynb


In [10]:
# generate test_X
test_X = []
no_face = []
with_face = []
#for video in tqdm(test_video_files[:32]):
for noface,video in tqdm(enumerate(test_video_files)):
    cou = detect_video(video)
    # if no face found
    if cou is None:
        no_face.append(noface)
        continue
    test_X.append(cou)
    with_face.append(noface)
    pass

# convert test_X to array
test_X = np.array(test_X)

no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found
no face found



In [11]:
!ls

__notebook__.ipynb


In [12]:
!mkdir test_images

In [13]:
%cd test_images

/kaggle/working/test_images


In [14]:
%mkdir image_class

In [15]:
%cd image_class

/kaggle/working/test_images/image_class


In [16]:
from PIL import Image
import numpy as np

for i,x in enumerate(test_X):
    w, h = 256, 256
    img = Image.fromarray(x, 'RGB')
    img.save(str(i)+ ".jpg" )
    img.show()


In [17]:
!ls

0.jpg	 135.jpg  172.jpg  209.jpg  246.jpg  283.jpg  32.jpg   357.jpg	67.jpg
1.jpg	 136.jpg  173.jpg  21.jpg   247.jpg  284.jpg  320.jpg  358.jpg	68.jpg
10.jpg	 137.jpg  174.jpg  210.jpg  248.jpg  285.jpg  321.jpg  359.jpg	69.jpg
100.jpg  138.jpg  175.jpg  211.jpg  249.jpg  286.jpg  322.jpg  36.jpg	7.jpg
101.jpg  139.jpg  176.jpg  212.jpg  25.jpg   287.jpg  323.jpg  360.jpg	70.jpg
102.jpg  14.jpg   177.jpg  213.jpg  250.jpg  288.jpg  324.jpg  361.jpg	71.jpg
103.jpg  140.jpg  178.jpg  214.jpg  251.jpg  289.jpg  325.jpg  362.jpg	72.jpg
104.jpg  141.jpg  179.jpg  215.jpg  252.jpg  29.jpg   326.jpg  363.jpg	73.jpg
105.jpg  142.jpg  18.jpg   216.jpg  253.jpg  290.jpg  327.jpg  37.jpg	74.jpg
106.jpg  143.jpg  180.jpg  217.jpg  254.jpg  291.jpg  328.jpg  38.jpg	75.jpg
107.jpg  144.jpg  181.jpg  218.jpg  255.jpg  292.jpg  329.jpg  39.jpg	76.jpg
108.jpg  145.jpg  182.jpg  219.jpg  256.jpg  293.jpg  33.jpg   4.jpg	77.jpg
109.jpg  146.jpg  183.jpg  22.jpg   257.jpg  294.jpg  330.jpg  40.jpg	78.jpg

In [18]:
len(no_face)

36

In [19]:
len(with_face)

364

In [20]:
%cd ..

/kaggle/working/test_images


In [21]:
%cd ..

/kaggle/working


In [22]:
model = MesoInception4()
model.load("/kaggle/input/meso-pretrain/MesoInception_DF")

In [23]:
from keras.preprocessing.image import ImageDataGenerator

image_data_generator = ImageDataGenerator(rescale=1.0/255)
data_generator = image_data_generator.flow_from_directory("test_images",classes=["image_class"],batch_size=364) 

""" Here test_images folder will contain another dolder having any name
    That folder will contain all the test images we will work on
"""

Found 364 images belonging to 1 classes.


' Here test_images folder will contain another dolder having any name\n    That folder will contain all the test images we will work on\n'

In [24]:
X,y = data_generator.next()

In [25]:
len(X)

364

In [26]:
probablistic_predictions = model.predict(X)

In [27]:
print(probablistic_predictions)

[[3.57530296e-01]
 [1.00000000e+00]
 [9.99046028e-01]
 [1.00000000e+00]
 [6.69506431e-01]
 [1.00000000e+00]
 [3.28371227e-02]
 [8.04284513e-01]
 [9.74353552e-01]
 [8.80600810e-01]
 [9.96803701e-01]
 [3.16811502e-02]
 [1.00000000e+00]
 [9.99997854e-01]
 [4.48970795e-02]
 [9.99980211e-01]
 [2.58901328e-01]
 [1.00000000e+00]
 [9.99999762e-01]
 [9.99999881e-01]
 [1.12329721e-02]
 [9.92709160e-01]
 [1.65021718e-02]
 [9.88470018e-01]
 [9.97613728e-01]
 [9.99987900e-01]
 [9.98224139e-01]
 [1.00000000e+00]
 [9.88342404e-01]
 [3.89740467e-02]
 [1.00000000e+00]
 [1.00000000e+00]
 [9.99817729e-01]
 [6.42713726e-01]
 [9.99997377e-01]
 [1.00000000e+00]
 [9.99922037e-01]
 [9.99414325e-01]
 [5.38041532e-01]
 [9.87967491e-01]
 [9.86591101e-01]
 [9.99832451e-01]
 [1.00000000e+00]
 [1.00000000e+00]
 [9.99990582e-01]
 [8.36002767e-01]
 [9.99967813e-01]
 [5.72889328e-01]
 [5.58596671e-01]
 [9.99854982e-01]
 [9.71654296e-01]
 [2.11377650e-01]
 [6.59201443e-01]
 [5.99537373e-01]
 [9.88856554e-01]
 [6.611403

In [28]:
import pandas as pd
sample_submission = pd.read_csv("../input/deepfake-detection-challenge/sample_submission.csv")

In [29]:
submission = pd.read_csv(os.path.join(INPUT, "sample_submission.csv"))
# fill in 0.5 for no face use
submission["label"] = 0.5
submission.head()

,filename,label
0,aassnaulhq.mp4,0.5
1,aayfryxljh.mp4,0.5
2,acazlolrpz.mp4,0.5
3,adohdulfwb.mp4,0.5
4,ahjnxtiamx.mp4,0.5


In [30]:
preds = probablistic_predictions

In [31]:
type(preds)

numpy.ndarray

In [32]:
type(submission)

pandas.core.frame.DataFrame

In [33]:
len(preds)

364

In [34]:
len(submission)

400

In [35]:
# convert submission to numpy.ndarray
sub = submission.values

In [36]:
len(sub)

400

In [37]:
with_face_a = np.asarray(with_face)

In [38]:
with_face_a

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  13,
        14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  32,  33,  35,  36,  37,  38,  39,  40,
        41,  43,  44,  46,  47,  48,  49,  50,  51,  52,  53,  55,  56,
        57,  58,  59,  60,  61,  62,  64,  65,  67,  68,  69,  70,  71,
        72,  73,  74,  75,  76,  77,  78,  79,  81,  82,  83,  84,  85,
        86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,  98,
        99, 100, 101, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112,
       113, 114, 115, 116, 117, 118, 120, 121, 122, 123, 124, 125, 126,
       127, 129, 130, 131, 132, 133, 134, 135, 136, 138, 139, 140, 141,
       142, 143, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 159, 160, 161, 162, 163, 164, 165, 166, 168, 169, 170,
       171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 182, 183, 184,
       185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 196, 19

In [39]:
for c in range(len(with_face)):
    x = with_face[c]
    sub[x][1] = preds[c][0]

In [40]:
len(sub)

400

In [41]:
type(sub)

numpy.ndarray

In [42]:
sub

array([['aassnaulhq.mp4', 0.3575303],
       ['aayfryxljh.mp4', 1.0],
       ['acazlolrpz.mp4', 0.999046],
       ['adohdulfwb.mp4', 1.0],
       ['ahjnxtiamx.mp4', 0.66950643],
       ['ajiyrjfyzp.mp4', 1.0],
       ['aktnlyqpah.mp4', 0.032837123],
       ['alrtntfxtd.mp4', 0.8042845],
       ['aomqqjipcp.mp4', 0.97435355],
       ['apedduehoy.mp4', 0.8806008],
       ['apvzjkvnwn.mp4', 0.9968037],
       ['aqrsylrzgi.mp4', 0.03168115],
       ['axfhbpkdlc.mp4', 0.5],
       ['ayipraspbn.mp4', 1.0],
       ['bcbqxhziqz.mp4', 0.99999785],
       ['bcvheslzrq.mp4', 0.04489708],
       ['bdshuoldwx.mp4', 0.9999802],
       ['bfdopzvxbi.mp4', 0.25890133],
       ['bfjsthfhbd.mp4', 1.0],
       ['bjyaxvggle.mp4', 0.99999976],
       ['bkcyglmfci.mp4', 0.9999999],
       ['bktkwbcawi.mp4', 0.011232972],
       ['bkuzquigyt.mp4', 0.99270916],
       ['blnmxntbey.mp4', 0.016502172],
       ['blszgmxkvu.mp4', 0.98847],
       ['bnuwxhfahw.mp4', 0.9976137],
       ['bofrwgeyjo.mp4', 0.9999879],

In [43]:
sample_submission

,filename,label
0,aassnaulhq.mp4,0
1,aayfryxljh.mp4,0
2,acazlolrpz.mp4,0
3,adohdulfwb.mp4,0
4,ahjnxtiamx.mp4,0
...,...,...
395,ztyvglkcsf.mp4,0
396,zuwwbbusgl.mp4,0
397,zxacihctqp.mp4,0
398,zyufpqvpyu.mp4,0


In [44]:
sub_df = pd.DataFrame({'filename': sub[:, 0], 'label': sub[:, 1]})

In [45]:
sub_df

,filename,label
0,aassnaulhq.mp4,0.35753
1,aayfryxljh.mp4,1
2,acazlolrpz.mp4,0.999046
3,adohdulfwb.mp4,1
4,ahjnxtiamx.mp4,0.669506
...,...,...
395,ztyvglkcsf.mp4,0.972413
396,zuwwbbusgl.mp4,0.488573
397,zxacihctqp.mp4,0.128708
398,zyufpqvpyu.mp4,0.999995


In [46]:
sub_df.to_csv("submission.csv", index=False)
sub_df.head()

,filename,label
0,aassnaulhq.mp4,0.35753
1,aayfryxljh.mp4,1
2,acazlolrpz.mp4,0.999046
3,adohdulfwb.mp4,1
4,ahjnxtiamx.mp4,0.669506
